In [1]:
%pip install --user "psycopg[binary]>=3.2,<4"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.5/215.5 kB 312.9 kB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 4.1 MB/s eta 0:00:000:00:0100:01
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import importlib.metadata as metadata
import psycopg

PROJECT = Path("/home/jovyan/project/georgia-bid-readiness")

DB_SETTINGS = {
    "host": "/home/jovyan/.gbr-pg-socket",
    "dbname": "georgia_bid_readiness",
    "user": "jovyan",
    "connect_timeout": 10,
}

with psycopg.connect(**DB_SETTINGS) as connection:
    with connection.cursor() as cursor:
        cursor.execute("""
            SELECT current_database(), current_user,
                   current_setting('server_version');
        """)
        database, user, version = cursor.fetchone()

print("Database:", database)
print("User:", user)
print("PostgreSQL version:", version)
print("Python connection: successful")

# Record the connector version without replacing existing requirements.
requirements = PROJECT / "requirements-cpu.txt"
existing = (
    requirements.read_text(encoding="utf-8")
    if requirements.exists()
    else ""
)

if not any(
    line.strip().lower().startswith("psycopg")
    for line in existing.splitlines()
):
    with requirements.open("a", encoding="utf-8") as file:
        if existing and not existing.endswith("\n"):
            file.write("\n")
        file.write(
            f"psycopg[binary]=={metadata.version('psycopg')}\n"
        )

Database: georgia_bid_readiness
User: jovyan
PostgreSQL version: 17.11
Python connection: successful


In [3]:
schema_sql = """
CREATE SCHEMA IF NOT EXISTS core;

CREATE TABLE IF NOT EXISTS core.solicitations (
    solicitation_id TEXT PRIMARY KEY,
    agency_name TEXT NOT NULL,
    solicitation_number TEXT NOT NULL,
    title TEXT NOT NULL,
    source_page_url TEXT NOT NULL,
    created_at TIMESTAMPTZ NOT NULL DEFAULT CURRENT_TIMESTAMP,
    UNIQUE (agency_name, solicitation_number)
);

CREATE TABLE IF NOT EXISTS core.documents (
    document_id TEXT PRIMARY KEY,
    solicitation_id TEXT NOT NULL
        REFERENCES core.solicitations(solicitation_id),
    document_type TEXT NOT NULL
        CHECK (document_type IN (
            'original', 'addendum', 'attachment',
            'bid_results', 'award_notice'
        )),
    amendment_number INTEGER,
    created_at TIMESTAMPTZ NOT NULL DEFAULT CURRENT_TIMESTAMP,
    CHECK (
        (document_type = 'addendum'
         AND amendment_number IS NOT NULL
         AND amendment_number > 0)
        OR
        (document_type <> 'addendum' AND amendment_number IS NULL)
    )
);

CREATE TABLE IF NOT EXISTS core.document_versions (
    document_id TEXT NOT NULL
        REFERENCES core.documents(document_id),
    sha256 TEXT NOT NULL
        CHECK (sha256 ~ '^[0-9a-f]{64}$'),
    local_path TEXT NOT NULL,
    size_bytes BIGINT NOT NULL CHECK (size_bytes > 0),
    page_count INTEGER NOT NULL CHECK (page_count > 0),
    created_at TIMESTAMPTZ NOT NULL DEFAULT CURRENT_TIMESTAMP,
    PRIMARY KEY (document_id, sha256)
);

CREATE TABLE IF NOT EXISTS core.pipeline_runs (
    run_id TEXT PRIMARY KEY,
    run_type TEXT NOT NULL
        CHECK (run_type IN ('ingestion', 'extraction')),
    parent_run_id TEXT REFERENCES core.pipeline_runs(run_id),
    status TEXT NOT NULL
        CHECK (status IN ('running', 'ok', 'partial_failure', 'failed')),
    started_at TIMESTAMPTZ,
    completed_at TIMESTAMPTZ,
    artifact_path TEXT NOT NULL,
    metadata JSONB NOT NULL DEFAULT '{}'::jsonb,
    CHECK (
        started_at IS NULL OR completed_at IS NULL
        OR completed_at >= started_at
    )
);

CREATE TABLE IF NOT EXISTS core.document_observations (
    run_id TEXT NOT NULL REFERENCES core.pipeline_runs(run_id),
    document_id TEXT NOT NULL REFERENCES core.documents(document_id),
    version_sha256 TEXT,
    source_url TEXT NOT NULL,
    source_filename TEXT NOT NULL,
    resolved_url TEXT,
    attempted_at TIMESTAMPTZ NOT NULL,
    retrieved_at TIMESTAMPTZ,
    http_status INTEGER,
    status TEXT NOT NULL CHECK (status IN ('ok', 'failed')),
    storage_action TEXT,
    error_message TEXT,
    PRIMARY KEY (run_id, document_id),
    FOREIGN KEY (document_id, version_sha256)
        REFERENCES core.document_versions(document_id, sha256),
    CHECK (status <> 'ok' OR version_sha256 IS NOT NULL)
);

CREATE TABLE IF NOT EXISTS core.page_extractions (
    extraction_run_id TEXT NOT NULL
        REFERENCES core.pipeline_runs(run_id),
    document_id TEXT NOT NULL,
    document_sha256 TEXT NOT NULL,
    page_number INTEGER NOT NULL CHECK (page_number > 0),
    extraction_method TEXT NOT NULL,
    extractor_version TEXT NOT NULL,
    status TEXT NOT NULL CHECK (status IN ('ok', 'failed')),
    raw_text TEXT,
    clean_text TEXT,
    character_count INTEGER CHECK (character_count >= 0),
    non_whitespace_characters INTEGER
        CHECK (non_whitespace_characters >= 0),
    quality_flags JSONB NOT NULL DEFAULT '[]'::jsonb,
    needs_review BOOLEAN NOT NULL,
    error_message TEXT,
    PRIMARY KEY (extraction_run_id, document_id, page_number),
    FOREIGN KEY (document_id, document_sha256)
        REFERENCES core.document_versions(document_id, sha256),
    CHECK (jsonb_typeof(quality_flags) = 'array'),
    CHECK (
        status <> 'ok'
        OR (raw_text IS NOT NULL AND clean_text IS NOT NULL)
    )
);

CREATE TABLE IF NOT EXISTS core.page_reviews (
    review_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    extraction_run_id TEXT NOT NULL,
    document_id TEXT NOT NULL,
    page_number INTEGER NOT NULL,
    review_status TEXT NOT NULL
        CHECK (review_status IN ('resolved', 'needs_followup')),
    resolution TEXT NOT NULL,
    ocr_required BOOLEAN NOT NULL,
    reviewed_by TEXT NOT NULL,
    reviewed_at TIMESTAMPTZ NOT NULL,
    review_method TEXT NOT NULL,
    notes TEXT NOT NULL,
    FOREIGN KEY (extraction_run_id, document_id, page_number)
        REFERENCES core.page_extractions(
            extraction_run_id, document_id, page_number
        ),
    UNIQUE (
        extraction_run_id, document_id, page_number,
        reviewed_at, reviewed_by
    )
);

CREATE INDEX IF NOT EXISTS idx_documents_solicitation
    ON core.documents(solicitation_id);

CREATE INDEX IF NOT EXISTS idx_observations_version
    ON core.document_observations(document_id, version_sha256);

CREATE INDEX IF NOT EXISTS idx_pages_version
    ON core.page_extractions(document_id, document_sha256);
"""

schema_path = PROJECT / "warehouse/ddl/001_core_schema.sql"
schema_path.parent.mkdir(parents=True, exist_ok=True)

# Preserve an existing schema file if its contents differ.
if schema_path.exists():
    if schema_path.read_text(encoding="utf-8") != schema_sql:
        raise RuntimeError(
            "A different schema file already exists. "
            "Review it before replacing or applying changes."
        )
else:
    schema_path.write_text(schema_sql, encoding="utf-8")

# Commit all statements together; roll back if one fails.
with psycopg.connect(**DB_SETTINGS) as connection:
    with connection.cursor() as cursor:
        cursor.execute(schema_sql)

print("Schema saved:", schema_path)
print("Foundation tables created.")

Schema saved: /home/jovyan/project/georgia-bid-readiness/warehouse/ddl/001_core_schema.sql
Foundation tables created.


In [4]:
with psycopg.connect(**DB_SETTINGS) as connection:
    with connection.cursor() as cursor:
        cursor.execute("""
            SELECT table_name
            FROM information_schema.tables
            WHERE table_schema = 'core'
              AND table_type = 'BASE TABLE'
            ORDER BY table_name;
        """)
        tables = [row[0] for row in cursor.fetchall()]

print("Tables found:", len(tables))
for table in tables:
    print("-", table)

Tables found: 7
- document_observations
- document_versions
- documents
- page_extractions
- page_reviews
- pipeline_runs
- solicitations


In [7]:
import json
from datetime import datetime, timezone
from psycopg import sql
from psycopg.types.json import Jsonb

SOLICITATION_ID = "commerce-ga-25-012"

# Use the extraction run that contains your completed page review.
EXTRACTION_DIR = (
    PROJECT / "data/processed/text" / SOLICITATION_ID
    / "20261016T225302199674Z"
)

summary_path = EXTRACTION_DIR / "extraction_summary.json"
pages_path = EXTRACTION_DIR / "pages.jsonl"
reviews_path = EXTRACTION_DIR / "page_reviews.jsonl"

def read_jsonl(path):
    return [
        json.loads(line)
        for line in path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]

summary = json.loads(summary_path.read_text(encoding="utf-8"))
pages = read_jsonl(pages_path)
reviews = read_jsonl(reviews_path)

# Load complete ingestion manifests for this solicitation.
ingestions = []

for path in sorted(
    (PROJECT / "data/processed/manifests").glob(
        f"{SOLICITATION_ID}_*.json"
    )
):
    data = json.loads(path.read_text(encoding="utf-8"))
    if data.get("packet_download_complete"):
        ingestions.append((path, data))

parent_path = (PROJECT / summary["input_manifest"]).resolve()
parent_matches = [
    data for path, data in ingestions
    if path.resolve() == parent_path
]

if len(parent_matches) != 1:
    raise RuntimeError("Extraction's parent ingestion manifest was not found.")

parent_run_id = parent_matches[0]["ingestion_run_id"]
extraction_run_id = summary["extraction_run_id"]

if len(pages) != summary["expected_pages"]:
    raise RuntimeError("Page records do not match the expected page count.")

if any(p["extraction_run_id"] != extraction_run_id for p in pages):
    raise RuntimeError("Page records contain an unexpected extraction run.")

if any(r["extraction_run_id"] != extraction_run_id for r in reviews):
    raise RuntimeError("Reviews contain an unexpected extraction run.")

# Confirm the review refers to the same file version as its page.
page_lookup = {
    (p["document_id"], p["page_number"]): p
    for p in pages
}

if len(page_lookup) != len(pages):
    raise RuntimeError("Duplicate page keys found in the extraction file.")

for review in reviews:
    page = page_lookup.get(
        (review["document_id"], review["page_number"])
    )
    if page is None or page["document_sha256"] != review["document_sha256"]:
        raise RuntimeError("A review does not match its extracted page.")

def insert_once(cursor, table, values):
    """Insert a record; preserve an existing record with the same key."""
    statement = sql.SQL("""
        INSERT INTO {} ({})
        VALUES ({})
        ON CONFLICT DO NOTHING
    """).format(
        sql.Identifier("core", table),
        sql.SQL(", ").join(sql.Identifier(key) for key in values),
        sql.SQL(", ").join(sql.Placeholder() for _ in values),
    )
    cursor.execute(statement, list(values.values()))

def timestamp_from_run_id(run_id):
    # Our run IDs encode UTC time; retain that meaning explicitly.
    return datetime.strptime(
        run_id, "%Y%m%dT%H%M%S%fZ"
    ).replace(tzinfo=timezone.utc)

with psycopg.connect(**DB_SETTINGS) as connection:
    with connection.cursor() as cursor:

        insert_once(cursor, "solicitations", {
            "solicitation_id": SOLICITATION_ID,
            "agency_name": "City of Commerce, Georgia",
            "solicitation_number": "25-012",
            "title": "Janitorial Services for City of Commerce Municipal Buildings",
            "source_page_url": parent_matches[0]["source_page"],
        })

        for manifest_file, ingestion in ingestions:
            ingestion_id = ingestion["ingestion_run_id"]

            insert_once(cursor, "pipeline_runs", {
                "run_id": ingestion_id,
                "run_type": "ingestion",
                "status": "ok",
                "started_at": timestamp_from_run_id(ingestion_id),
                "completed_at": ingestion.get("completed_at_utc"),
                "artifact_path": manifest_file.relative_to(PROJECT).as_posix(),
                "metadata": Jsonb({
                    "expected_document_count":
                        ingestion["expected_document_count"],
                    "source_listing_sha256":
                        ingestion.get("source_listing_sha256"),
                }),
            })

            for record in ingestion["records"]:
                if record["status"] != "ok":
                    raise RuntimeError(
                        "A complete manifest contains a failed document."
                    )

                insert_once(cursor, "documents", {
                    "document_id": record["document_id"],
                    "solicitation_id": record["solicitation_id"],
                    "document_type": record["document_type"],
                    "amendment_number": record["amendment_number"],
                })

                insert_once(cursor, "document_versions", {
                    "document_id": record["document_id"],
                    "sha256": record["sha256"],
                    "local_path": record["local_path"],
                    "size_bytes": record["size_bytes"],
                    "page_count": record["page_count"],
                })

                insert_once(cursor, "document_observations", {
                    "run_id": ingestion_id,
                    "document_id": record["document_id"],
                    "version_sha256": record["sha256"],
                    "source_url": record["source_url"],
                    "source_filename": record["source_filename"],
                    "resolved_url": record.get("resolved_url"),
                    "attempted_at": record["attempted_at_utc"],
                    "retrieved_at": record["retrieved_at_utc"],
                    "http_status": record.get("http_status"),
                    "status": "ok",
                    "storage_action": record["storage_action"],
                })

        insert_once(cursor, "pipeline_runs", {
            "run_id": extraction_run_id,
            "run_type": "extraction",
            "parent_run_id": parent_run_id,
            "status": (
                "ok" if summary["all_pages_extracted"]
                else "partial_failure"
            ),
            "started_at": timestamp_from_run_id(extraction_run_id),
            "completed_at": summary["completed_at_utc"],
            "artifact_path": summary_path.relative_to(PROJECT).as_posix(),
            "metadata": Jsonb({
                "extractor": summary["extractor"],
                "extractor_version": summary["extractor_version"],
                "low_text_threshold": summary["low_text_threshold"],
                "expected_pages": summary["expected_pages"],
            }),
        })

        for page in pages:
            insert_once(cursor, "page_extractions", {
                "extraction_run_id": page["extraction_run_id"],
                "document_id": page["document_id"],
                "document_sha256": page["document_sha256"],
                "page_number": page["page_number"],
                "extraction_method": page["extraction_method"],
                "extractor_version": page["extractor_version"],
                "status": page["status"],
                "raw_text": page["raw_text"],
                "clean_text": page["clean_text"],
                "character_count": page["character_count"],
                "non_whitespace_characters":
                    page["non_whitespace_characters"],
                "quality_flags": Jsonb(page["quality_flags"]),
                "needs_review": page["needs_review"],
                "error_message": page.get("error"),
            })

        for review in reviews:
            insert_once(cursor, "page_reviews", {
                key: review[key]
                for key in [
                    "extraction_run_id", "document_id", "page_number",
                    "review_status", "resolution", "ocr_required",
                    "reviewed_by", "reviewed_at_utc",
                    "review_method", "notes",
                ]
                if key != "reviewed_at_utc"
            } | {"reviewed_at": review["reviewed_at_utc"]})

print("Warehouse load committed successfully.")
print("Complete ingestion runs loaded:", len(ingestions))
print("Extraction run:", extraction_run_id)
print("Input page records:", len(pages))
print("Input review records:", len(reviews))

Warehouse load committed successfully.
Complete ingestion runs loaded: 2
Extraction run: 20261016T225302199674Z
Input page records: 28
Input review records: 1


In [6]:
tables = [
    "solicitations",
    "documents",
    "document_versions",
    "pipeline_runs",
    "document_observations",
    "page_extractions",
    "page_reviews",
]

def warehouse_counts():
    counts = {}
    with psycopg.connect(**DB_SETTINGS) as connection:
        with connection.cursor() as cursor:
            for table in tables:
                cursor.execute(
                    sql.SQL("SELECT COUNT(*) FROM {}").format(
                        sql.Identifier("core", table)
                    )
                )
                counts[table] = cursor.fetchone()[0]
    return counts

first_counts = warehouse_counts()

for table, count in first_counts.items():
    print(f"{table}: {count}")

solicitations: 1
documents: 12
document_versions: 12
pipeline_runs: 3
document_observations: 24
page_extractions: 28
page_reviews: 1


In [8]:
second_counts = warehouse_counts()

assert first_counts == second_counts, (
    f"Counts changed: {first_counts} -> {second_counts}"
)

print("Repeat-load check passed: no duplicate records added.")

Repeat-load check passed: no duplicate records added.


In [9]:
with psycopg.connect(**DB_SETTINGS) as connection:
    with connection.cursor() as cursor:
        cursor.execute("""
            SELECT
                p.document_id,
                p.page_number,
                p.quality_flags,
                r.review_status,
                r.resolution,
                r.ocr_required
            FROM core.page_extractions AS p
            LEFT JOIN LATERAL (
                SELECT
                    review_status,
                    resolution,
                    ocr_required
                FROM core.page_reviews
                WHERE extraction_run_id = p.extraction_run_id
                  AND document_id = p.document_id
                  AND page_number = p.page_number
                ORDER BY reviewed_at DESC, review_id DESC
                LIMIT 1
            ) AS r ON TRUE
            WHERE p.extraction_run_id = %s
              AND p.needs_review = TRUE;
        """, (extraction_run_id,))

        for row in cursor.fetchall():
            print(row)

('commerce-ga-25-012-original', 10, ['low_text'], 'resolved', 'legitimate_low_text_page', False)


In [10]:
from pathlib import Path
from datetime import datetime, timezone
import json
import subprocess

import psycopg
from psycopg import sql

PROJECT = Path("/home/jovyan/project/georgia-bid-readiness")
PG_BIN = Path("/home/jovyan/.conda/envs/gbr-postgres/bin")
SOCKET = "/home/jovyan/.gbr-pg-socket"
SOURCE_DB = "georgia_bid_readiness"

BACKUP_DIR = PROJECT / "data/processed/database_backups"
backups = sorted(
    BACKUP_DIR.glob("georgia_bid_readiness_*.dump"),
    key=lambda path: path.stat().st_mtime,
)

if not backups:
    raise FileNotFoundError(
        "No database export found. Run the pg_dump command first."
    )

backup = backups[-1]
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%f")
RESTORE_DB = f"gbr_restore_check_{stamp.lower()}"

def run_pg(tool, *arguments):
    result = subprocess.run(
        [str(PG_BIN / tool), *arguments],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(
            f"{tool} failed:\n{result.stderr or result.stdout}"
        )
    return result

print("Backup:", backup.name)
print("Test database:", RESTORE_DB)

# Create a new, separate database.
run_pg(
    "createdb",
    "-h", SOCKET,
    "-U", "jovyan",
    "--template=template0",
    RESTORE_DB,
)

# Restore in a single transaction; stop on any error.
run_pg(
    "pg_restore",
    "-h", SOCKET,
    "-U", "jovyan",
    "-d", RESTORE_DB,
    "--exit-on-error",
    "--single-transaction",
    str(backup),
)

print("Restore completed.")

tables = [
    "solicitations",
    "documents",
    "document_versions",
    "pipeline_runs",
    "document_observations",
    "page_extractions",
    "page_reviews",
]

def read_table(database, table):
    with psycopg.connect(
        host=SOCKET,
        dbname=database,
        user="jovyan",
        connect_timeout=10,
    ) as connection:
        with connection.cursor() as cursor:
            # Compare every stored field, not just row counts.
            cursor.execute(
                sql.SQL("""
                    SELECT to_jsonb(t)::text
                    FROM {} AS t
                    ORDER BY 1
                """).format(sql.Identifier("core", table))
            )
            return [row[0] for row in cursor.fetchall()]

checks = []

for table in tables:
    source_rows = read_table(SOURCE_DB, table)
    restored_rows = read_table(RESTORE_DB, table)
    matches = source_rows == restored_rows

    checks.append({
        "table": table,
        "source_rows": len(source_rows),
        "restored_rows": len(restored_rows),
        "contents_match": matches,
    })

    print(
        f"{table}: source={len(source_rows)}, "
        f"restored={len(restored_rows)}, "
        f"{'PASS' if matches else 'MISMATCH'}"
    )

passed = all(check["contents_match"] for check in checks)

report = {
    "checked_at_utc": datetime.now(timezone.utc).isoformat(),
    "backup_file": backup.relative_to(PROJECT).as_posix(),
    "source_database": SOURCE_DB,
    "restore_database": RESTORE_DB,
    "all_table_contents_match": passed,
    "checks": checks,
}

report_path = PROJECT / "reports" / f"database_restore_check_{stamp}.json"
report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")

print("\nVerification report:", report_path)

if not passed:
    raise RuntimeError(
        "Restored contents differ from the current database. "
        "The backup may predate recent changes; inspect the results."
    )

print("RESTORE CHECK PASSED: all seven tables match exactly.")

Backup: georgia_bid_readiness_20261017T152937.dump
Test database: gbr_restore_check_20261017t153321887561
Restore completed.
solicitations: source=1, restored=1, PASS
documents: source=12, restored=12, PASS
document_versions: source=12, restored=12, PASS
pipeline_runs: source=3, restored=3, PASS
document_observations: source=24, restored=24, PASS
page_extractions: source=28, restored=28, PASS
page_reviews: source=1, restored=1, PASS

Verification report: /home/jovyan/project/georgia-bid-readiness/reports/database_restore_check_20261017T153321887561.json
RESTORE CHECK PASSED: all seven tables match exactly.
